# ELIQSIR — Step 1: Data Extraction

This notebook runs each extractor **independently** and saves the raw output to `data/raw/` as Parquet files that feed `02_transformation_and_loading.ipynb`.

| Step | Source | What is fetched |
|------|--------|-----------------|
| 0 | — | Pre-flight checks (env vars, MySQL connectivity, dump file) |
| 1 | — | Setup: paths & logging |
| 2 | **UniProt** | All reviewed human proteins (Swiss-Prot) |
| 3 | **ChEMBL** | `setup_database()` → bioactivity records for those proteins |
| 4 | **PDBe** | Best 3-D structures for proteins with ChEMBL activity |
| 5 | **PubMed** | Article abstracts for publications cited in ChEMBL |
| 6 | — | Extraction summary |

> **Run cells top-to-bottom.** Each section is independent enough to be re-run alone after a failure — Parquet files from previous sections are loaded from disk if they exist.


## 0 · Pre-flight Checks

Quickly verify that all required settings are present and that both MySQL databases are reachable **before** starting any long-running extraction.


In [1]:
import sys
from pathlib import Path

# ── make src/ importable without pip install ─────────────────────────────────
REPO_ROOT = Path("..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import mysql.connector

from src.config import settings
from src.extraction.chembl_extractor import discover_chembl_database
from src.utils.logging_config import get_logger

logger = get_logger("notebook.extraction")

# Shorthand for safe display of paths (project-relative, never absolute)
_dp = settings.display_path

RAW_DIR = REPO_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# ── 1. Required env vars ──────────────────────────────────────────────────────
checks = {
    "NCBI_EMAIL set":       bool(settings.ncbi_email),
    "MYSQL_USER set":       bool(settings.mysql_user),
    "MYSQL_PASSWORD set":   bool(settings.mysql_password),
}

# ── 2. ChEMBL SQLite database discovery ───────────────────────────────────────
chembl_version = None
chembl_db_path = None
try:
    chembl_version, chembl_db_path = discover_chembl_database(settings.chembl_dir)
    checks[f"ChEMBL SQLite v{chembl_version} found"] = True
except FileNotFoundError as exc:
    checks["ChEMBL SQLite database found"] = False
    chembl_msg = str(exc)

# ── 3. MySQL connectivity (warehouse only) ────────────────────────────────────
def _check_mysql(database=None) -> tuple[bool, str]:
    cfg = dict(
        host=settings.mysql_host,
        port=settings.mysql_port,
        user=settings.mysql_user,
        password=settings.mysql_password,
    )
    if database:
        cfg["database"] = database
    try:
        conn = mysql.connector.connect(**cfg)
        conn.close()
        return True, "OK"
    except Exception as exc:
        return False, str(exc)

mysql_ok, mysql_msg         = _check_mysql()
warehouse_ok, warehouse_msg = _check_mysql(settings.mysql_db)

checks["MySQL server reachable"]                     = mysql_ok
checks[f"Warehouse DB '{settings.mysql_db}' exists"] = warehouse_ok

# ── 4. Report ──────────────────────────────────────────────────────────────────
all_ok = True
for label, passed in checks.items():
    icon = "✅" if passed else "❌"
    print(f"  {icon}  {label}")
    if not passed:
        all_ok = False

if not all_ok:
    print("\n⚠  Fix the items above before running the extractors.")
    print(f"   MySQL server msg  : {mysql_msg}")
    print(f"   Warehouse DB msg  : {warehouse_msg}")
    if chembl_db_path is None:
        print(f"   ChEMBL dir        : {_dp(settings.chembl_dir)}")
        print("   → Download SQLite version from:")
        print("     https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/")
    print(f"   NCBI email        : {'Present' if settings.ncbi_email else 'NOT SET'}")
else:
    print("\n✅  All checks passed — ready to extract.")
    print(f"\n   ChEMBL version    : {chembl_version}")
    print(f"   ChEMBL database   : {_dp(chembl_db_path)}")
    print(f"   MySQL host        : {settings.mysql_host}:{settings.mysql_port}")
    print(f"   Warehouse database: {settings.mysql_db}")
    print(f"   Raw output dir    : {_dp(RAW_DIR)}")
    print(f"   NCBI email        : {'Present' if settings.ncbi_email else 'NOT SET'}")

  ✅  NCBI_EMAIL set
  ✅  MYSQL_USER set
  ✅  MYSQL_PASSWORD set
  ✅  ChEMBL SQLite v36 found
  ✅  MySQL server reachable
  ❌  Warehouse DB 'eliqsir_dw' exists

⚠  Fix the items above before running the extractors.
   MySQL server msg  : OK
   Warehouse DB msg  : 1049 (42000): Unknown database 'eliqsir_dw'
   NCBI email        : Present


## 1 · Setup

Imports, paths, and logging — already done in the pre-flight cell above.  
This section is kept for reference; nothing extra to run.

---

## 2 · UniProt Extraction

`UniProtExtractor` queries the UniProt `/stream` endpoint and returns all **reviewed** (*Swiss-Prot*) human proteins in a single TSV request.

**Selection criteria:**
- Organism: *Homo sapiens* (taxonomy ID `9606`)
- Quality: reviewed (Swiss-Prot) only — manually curated, high confidence

**Columns returned:** `accession` · `gene_names` · `protein_name` · `organism_name` · `protein_sequence` · `protein_class` · `ec_number` · `catalyzed_reaction`


In [2]:
from src.extraction import UniProtExtractor

uniprot_ext = UniProtExtractor(
    organism_id=settings.uniprot_organism_id,   # 9606 = Homo sapiens
    reviewed=settings.uniprot_reviewed_only,    # Swiss-Prot only
)

df_raw_uniprot = uniprot_ext.extract()

print(f"Shape  : {df_raw_uniprot.shape}")
print(f"Columns: {list(df_raw_uniprot.columns)}")
display(df_raw_uniprot.head(5))


2026-03-26 04:05:20  INFO      src.extraction.uniprot_extractor  Fetching UniProt data – organism_id=9606, reviewed=True
2026-03-26 04:05:31  INFO      src.extraction.uniprot_extractor  UniProt extraction complete – 20431 proteins retrieved.
Shape  : (20431, 8)
Columns: ['accession', 'gene_names', 'protein_name', 'organism_name', 'protein_sequence', 'protein_class', 'ec_number', 'catalyzed_reaction']


,accession,gene_names,protein_name,organism_name,protein_sequence,protein_class,ec_number,catalyzed_reaction
0,A0A087X1C5,CYP2D7,Cytochrome P450 2D7 (EC 1.14.14.1),Homo sapiens (Human),MGLEALVPLAMIVAIFLLLVDLMHRHQRWAARYPPGPLPLPGLGNL...,Cytochrome P450 family,1.14.14.1,CATALYTIC ACTIVITY: Reaction=an organic molecu...
1,A0A096LP01,SMIM26 LINC00493,Small integral membrane protein 26,Homo sapiens (Human),MYRNEFTAWYRRMSVVYGIGTWSVLGSLLYYSRTMAKSSVDQKDGS...,SMIM26 family,NaN,NaN
2,A0A0B4J2F0,PIGBOS1,Protein PIGBOS1 (PIGB opposite strand protein 1),Homo sapiens (Human),MFRRLTFAQLLFATVLGIAGGVYIFQPVFEQYAKDQKELKEKMQLV...,NaN,NaN,NaN
3,A0A0C5B5G6,MT-RNR1,Mitochondrial-derived peptide MOTS-c (Mitochon...,Homo sapiens (Human),MRWQEMGYIFYPRKLR,NaN,NaN,NaN
4,A0A0K2S4Q6,CD300H,Protein CD300H (CD300 antigen-like family memb...,Homo sapiens (Human),MTQRAGAAMLPSALLLLCVPGCLTVSGPSTVMGAVGESLSVQCRYE...,CD300 family,NaN,NaN


In [3]:
out_path = RAW_DIR / "raw_uniprot.parquet"
df_raw_uniprot.to_parquet(out_path, index=False)
print(f"✓  Saved {len(df_raw_uniprot):,} rows → {_dp(out_path)}")


✓  Saved 20,431 rows → data/raw/raw_uniprot.parquet


## 3 · ChEMBL Bioactivity Data

Queries a **local SQLite** instance of [ChEMBL](https://www.ebi.ac.uk/chembl/) for bioactivity records
that link drugs to the UniProt proteins identified above.

The `ChemblExtractor` **auto-discovers** the ChEMBL version from the folder structure:
```
data/ChEMBL/
└── chembl_XX/                    # ← version detected from folder name
    └── chembl_XX_sqlite/
        └── chembl_XX.db          # ← SQLite database
```

**Download instructions:**
1. Go to https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/
2. Download `chembl_XX_sqlite.tar.gz` (~1.5 GB)
3. Extract and move the `chembl_XX` folder into `data/ChEMBL/`

**Caching:** Results are cached as Parquet files in `data/ChEMBL/cache/` for fast subsequent loads.

Columns extracted (23 total):

| Column | Description |
|--------|-------------|
| `activity_id` | ChEMBL internal activity identifier |
| `drug_chembl_id` | ChEMBL compound ID (e.g. `CHEMBL25`) |
| `drug_name` | Preferred compound name |
| `molecule_type` | Small molecule / Antibody / etc. |
| `molecular_weight` | Molecular weight (g/mol) |
| `canonical_smiles` | Canonical SMILES string |
| `target_chembl_id` | ChEMBL target ID |
| `target_name` | Target preferred name |
| `organism` | Target organism |
| `standard_type` | Measurement type (IC50, Ki, …) |
| `standard_value` | Numeric measurement value |
| `standard_units` | Units (nM, µM, …) |
| `pchembl_value` | −log₁₀(activity) |
| `assay_type` | Assay type code |
| `assay_description` | Free-text assay description |
| `assay_organism` | Organism used in assay |
| `confidence_score` | Target assignment confidence (0–9) |
| `article_title` | Source article title |
| `journal` | Journal name |
| `year` | Publication year |
| `pubmed_id` | PubMed ID of source article |
| `doi` | DOI of source article |
| `uniprot_id` | UniProt accession (join key) |

In [ ]:
# Reload modules to pick up code changes (dev convenience)
import importlib
import src.utils.logging_config
import src.extraction.chembl_extractor
import src.extraction

importlib.reload(src.utils.logging_config)
importlib.reload(src.extraction.chembl_extractor)
importlib.reload(src.extraction)

from src.extraction import ChemblExtractor

uniprot_accessions = df_raw_uniprot["accession"].dropna().unique().tolist()
print(f"Filtering ChEMBL for {len(uniprot_accessions):,} UniProt accessions …\n")

# Initialize extractor - auto-discovers ChEMBL version from folder structure
chembl_ext = ChemblExtractor(
    chembl_dir=settings.chembl_dir,
    use_cache=True,  # Cache results as Parquet for fast subsequent loads
)

# Optional: Print database summary
chembl_ext.print_database_info()

# Extract bioactivity data (with progress bar and caching)
df_raw_chembl = chembl_ext.extract(uniprot_ids=uniprot_accessions)

print(f"\nShape   : {df_raw_chembl.shape}")
print(f"Columns : {list(df_raw_chembl.columns)}")
display(df_raw_chembl.head(5))

# Save to raw output directory (separate from cache)
out_path = RAW_DIR / "raw_chembl.parquet"
df_raw_chembl.to_parquet(out_path, index=False)
print(f"\n✓  Saved {len(df_raw_chembl):,} rows → {_dp(out_path)}")

Filtering ChEMBL for 20,431 UniProt accessions …

2026-03-26 04:05:31  INFO      src.extraction.chembl_extractor  ============================================================
2026-03-26 04:05:31  INFO      src.extraction.chembl_extractor  ChEMBL SQLite Extractor initialized
2026-03-26 04:05:31  INFO      src.extraction.chembl_extractor    Version  : 36
2026-03-26 04:05:31  INFO      src.extraction.chembl_extractor    Database : data/ChEMBL/chembl_36/chembl_36_sqlite/chembl_36.db
2026-03-26 04:05:31  INFO      src.extraction.chembl_extractor    Cache dir: data/ChEMBL/cache
2026-03-26 04:05:31  INFO      src.extraction.chembl_extractor    Use cache: True
2026-03-26 04:05:31  INFO      src.extraction.chembl_extractor  ============================================================
✓ ChEMBL version 36 detected
  Database: data/ChEMBL/chembl_36/chembl_36_sqlite/chembl_36.db
  Size: 27.70 GB

ChEMBL v36 Database Summary
Database file: data/ChEMBL/chembl_36/chembl_36_sqlite/chembl_36.db
Size: 27

In [ ]:
# Summary statistics (already printed by extractor, but more detail here)
print("Unique drugs      :", df_raw_chembl["drug_chembl_id"].nunique())
print("Unique proteins   :", df_raw_chembl["uniprot_id"].nunique())
print("Unique PubMed IDs :", df_raw_chembl["pubmed_id"].nunique())
print("Articles with DOI :", df_raw_chembl["doi"].notna().sum())

print("\nMolecule types:")
display(df_raw_chembl["molecule_type"].value_counts().to_frame())

print("\nMeasurement types (top 10):")
display(df_raw_chembl["standard_type"].value_counts().head(10).to_frame())

# Save to raw output directory (separate from cache)
out_path = RAW_DIR / "raw_chembl.parquet"
df_raw_chembl.to_parquet(out_path, index=False)
print(f"\n✓  Saved {len(df_raw_chembl):,} rows → {_dp(out_path)}")

## 4 · PDBe Extraction

`PdbeExtractor` queries the **PDBe Graph API** `/best_structures/{uniprot_id}` endpoint for each protein that appeared in the ChEMBL results.

The `/best_structures/` endpoint returns pre-ranked, high-quality structures — no need to filter by resolution manually.

**Key columns returned:** `uniprot_id` · `pdb_id` · `chain_id` · `resolution` · `coverage` · `method` · `unp_start` · `unp_end`

**Rate limiting:** `PDBE_REQUEST_DELAY_S` (default `0.1` s between requests) is applied to comply with EBI server guidelines.


In [ ]:
from src.extraction import PdbeExtractor

# Only request structures for proteins that actually have ChEMBL activities
active_accessions = df_raw_chembl["uniprot_id"].dropna().unique().tolist()
print(f"Fetching PDB structures for {len(active_accessions):,} proteins …")
print("(This may take several minutes — 0.1 s rate-limit per request)")

pdbe_ext = PdbeExtractor(
    request_delay_s=settings.pdbe_request_delay_s,
    timeout_s=settings.pdbe_timeout_s,
)
df_raw_pdbe = pdbe_ext.extract(uniprot_ids=active_accessions)

print(f"\nShape   : {df_raw_pdbe.shape}")
print(f"Columns : {list(df_raw_pdbe.columns)}")
display(df_raw_pdbe.head(10))


In [ ]:
# Coverage & resolution overview
print("Proteins with at least one structure :", df_raw_pdbe["uniprot_id"].nunique())
print("Unique PDB entries                   :", df_raw_pdbe["pdb_id"].nunique())
print("\nExperimental methods:")
display(df_raw_pdbe["method"].value_counts().to_frame())

print("\nResolution (Å) — lower is better:")
display(df_raw_pdbe["resolution"].describe().to_frame().T)

# Save
out_path = RAW_DIR / "raw_pdbe.parquet"
df_raw_pdbe.to_parquet(out_path, index=False)
print(f"\n✓  Saved {len(df_raw_pdbe):,} rows → {_dp(out_path)}")


## 5 · PubMed Extraction

`PubMedExtractor` uses **Biopython's `Bio.Entrez`** to batch-fetch article metadata from NCBI PubMed via the E-utils XML API.

**Batching strategy:** IDs are sent in groups of `NCBI_BATCH_SIZE` (default `200`) to stay within NCBI limits and keep XML responses manageable.

**Columns returned (7 total):**

| Column | Description |
|--------|-------------|
| `pubmed_id` | PubMed article identifier |
| `abstract` | Full article abstract text |
| `authors` | Semicolon-separated author list (`Last, First`) |
| `pub_date` | Publication date string |
| `year` | Publication year (integer) |
| `month` | Publication month (integer) |
| `doi` | Digital Object Identifier |


> `NCBI_EMAIL` must be set in `.env` — NCBI requires an email address to identify API callers.


In [ ]:
from src.extraction import PubMedExtractor

# Use only the PubMed IDs that appeared in the ChEMBL results
pmids = (
    df_raw_chembl["pubmed_id"]
    .dropna()
    .unique()
    .tolist()
)
# Cast to str — Entrez expects string IDs
pmids = [str(int(p)) for p in pmids]
print(f"Fetching abstracts for {len(pmids):,} PubMed articles …")

pubmed_ext = PubMedExtractor(
    email=settings.ncbi_email,
    batch_size=settings.ncbi_batch_size,
    request_delay_s=settings.ncbi_request_delay_s,
)
df_raw_pubmed = pubmed_ext.extract(pubmed_ids=pmids)

print(f"\nShape   : {df_raw_pubmed.shape}")
print(f"Columns : {list(df_raw_pubmed.columns)}")
display(df_raw_pubmed.head(10))


In [ ]:
abstracts_found = df_raw_pubmed["abstract"].notna() & (df_raw_pubmed["abstract"].str.strip() != "")
print(f"Articles with non-empty abstract : {abstracts_found.sum():,} / {len(df_raw_pubmed):,}")
print(f"Average abstract length (chars)  : {df_raw_pubmed['abstract'].dropna().str.len().mean():.0f}")
print(f"Articles with authors            : {(df_raw_pubmed['authors'] != '').sum():,}")
print(f"Articles with DOI                : {(df_raw_pubmed['doi'] != '').sum():,}")
print(f"Articles with pub_date           : {(df_raw_pubmed['pub_date'] != '').sum():,}")

print("\nYear distribution (top 10):")
display(df_raw_pubmed["year"].value_counts().head(10).sort_index().to_frame())

# Save
out_path = RAW_DIR / "raw_pubmed.parquet"
df_raw_pubmed.to_parquet(out_path, index=False)
print(f"\n✓  Saved {len(df_raw_pubmed):,} rows → {_dp(out_path)}")


## 6 · Extraction Summary

A consolidated view of what was extracted from each source.


In [ ]:
summary = pd.DataFrame([
    {"source": "UniProt",  "rows": len(df_raw_uniprot), "file": "raw/raw_uniprot.parquet"},
    {"source": "ChEMBL",   "rows": len(df_raw_chembl),  "file": "raw/raw_chembl.parquet"},
    {"source": "PDBe",     "rows": len(df_raw_pdbe),    "file": "raw/raw_pdbe.parquet"},
    {"source": "PubMed",   "rows": len(df_raw_pubmed),  "file": "raw/raw_pubmed.parquet"},
])
display(summary.style.set_caption("Extraction Summary").hide(axis="index"))

print("\n✓  All raw Parquet files written to data/raw/")
print("   → Continue in notebooks/02_transformation_and_loading.ipynb")
